<a href="https://colab.research.google.com/github/JayTea704/MIS-433-Python-for-Predictive-Data-Analytics-Projects/blob/main/Band_Reccomendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MIS 433 Vibe Coding Challenge: Band Recommendation System

Type in a band you like, and this notebook recommends other bands you are likely to enjoy.

**How it works**

1. **Last.fm API** looks up the band and its genre tags, then pulls its *similar artists*. Last.fm builds this list from real listening data: it measures how much the audiences of two artists overlap, so a high match means many people who play your band also play that artist.
2. **OpenAI API** reviews the top 40 listener matches, picks the best mix of recommendations, and writes a one-sentence reason for each pick.
3. **Last.fm API** pulls listener counts and tags for each pick so you can see how popular it is and what it sounds like.

The final score blends the two signals: 60% listener overlap from Last.fm and 40% OpenAI's rating of how well the sound and style fit.

**Setup (one time)**

1. Get a free Last.fm API key at [last.fm/api/account/create](https://www.last.fm/api/account/create). Only the **API key** is needed, not the shared secret.
2. Get an OpenAI API key at [platform.openai.com/api-keys](https://platform.openai.com/api-keys).
3. In Colab, click the **key icon (Secrets)** in the left sidebar and add two secrets, with notebook access turned on:
   `LASTFM_API_KEY` and `OPENAI_API_KEY`
4. Run all cells (**Runtime > Run all**) and use the search box at the bottom.

In [1]:
# Step 1: Install and import libraries
!pip install -q --upgrade openai

import html, json, re, time, unicodedata
from urllib.parse import quote
import requests
import pandas as pd
from IPython.display import display, HTML
from openai import OpenAI

In [2]:
# Step 2: Load API keys from Colab Secrets (falls back to a hidden prompt)
from getpass import getpass

def load_secret(name):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass
    return getpass(f"Enter {name}: ").strip()

LASTFM_API_KEY = load_secret("LASTFM_API_KEY")
OPENAI_API_KEY = load_secret("OPENAI_API_KEY")

openai_client = OpenAI(api_key=OPENAI_API_KEY)

# The first model that works on your account is used. Edit this list if needed.
OPENAI_MODELS = ["gpt-5.4-mini", "gpt-5-mini", "gpt-4o-mini"]
print("Keys loaded.")

Keys loaded.


In [3]:
# Step 3: Last.fm API client
LASTFM_URL = "https://ws.audioscrobbler.com/2.0/"
HEADERS = {"User-Agent": "MIS433-BandRecommender/1.0"}

class LastFMError(Exception):
    pass

def lastfm(method, **params):
    """Call a Last.fm API method and return the JSON response."""
    params.update({"method": method, "api_key": LASTFM_API_KEY, "format": "json"})
    for attempt in range(3):
        r = requests.get(LASTFM_URL, params=params, headers=HEADERS, timeout=15)
        if r.status_code in (429, 500, 502, 503) and attempt < 2:
            time.sleep(2 * (attempt + 1))   # busy or rate limited: wait and retry
            continue
        try:
            data = r.json()
        except ValueError:
            raise LastFMError(f"Last.fm returned an unreadable response ({r.status_code}).")
        if "error" in data:
            if data["error"] in (10, 26):
                raise LastFMError("Last.fm rejected the API key. Check LASTFM_API_KEY in Colab Secrets.")
            raise LastFMError(data.get("message", f"Last.fm error {data['error']}"))
        time.sleep(0.2)   # stay well under Last.fm's rate limit
        return data
    raise LastFMError("Last.fm is busy. Try again in a minute.")

def as_list(value):
    """Last.fm returns a dict instead of a list when there is only one result."""
    if not value:
        return []
    return value if isinstance(value, list) else [value]

# Quick connection test
test = lastfm("artist.getInfo", artist="Radiohead")
print("Last.fm connected:", test["artist"]["name"])

Last.fm connected: Radiohead


In [4]:
# Step 4: Look up a band and the artists its listeners also play
def normalize(name):
    # lowercase, strip accents (Björk -> bjork), drop a leading "The" and punctuation
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode()
    name = re.sub(r"^the\s+", "", name.lower().strip())
    return re.sub(r"[^a-z0-9]+", "", name)

def get_artist_info(name):
    """Name, tags, listener count, and page link for one artist (autocorrects spelling)."""
    a = lastfm("artist.getInfo", artist=name, autocorrect=1)["artist"]
    return {
        "name": a["name"],
        "url": a.get("url", ""),
        "listeners": int(a.get("stats", {}).get("listeners", 0) or 0),
        "tags": [t["name"] for t in as_list(a.get("tags", {}).get("tag"))],
    }

def find_artist(name):
    """Find the band the user typed. Tries an exact lookup first, then a search."""
    try:
        return get_artist_info(name)
    except LastFMError:
        pass
    matches = as_list(lastfm("artist.search", artist=name, limit=5)
                      .get("results", {}).get("artistmatches", {}).get("artist"))
    return get_artist_info(matches[0]["name"]) if matches else None

def get_similar_artists(name, limit=50):
    """Artists whose listeners overlap with this one. match runs from 0 to 1."""
    data = lastfm("artist.getSimilar", artist=name, limit=limit, autocorrect=1)
    return [{"name": a["name"], "match": float(a.get("match", 0)), "url": a.get("url", "")}
            for a in as_list(data.get("similarartists", {}).get("artist"))]

In [5]:
# Step 5: OpenAI picks and explains the recommendations
def ask_openai_json(system, user):
    """Send a prompt and get back parsed JSON. Tries each model in OPENAI_MODELS."""
    last_error = None
    for model in OPENAI_MODELS:
        try:
            resp = openai_client.chat.completions.create(
                model=model,
                response_format={"type": "json_object"},
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": user}],
            )
            return json.loads(resp.choices[0].message.content)
        except Exception as e:
            last_error = e
            if "model" in str(e).lower():   # model not available: try the next one
                continue
            raise
    raise RuntimeError(f"No OpenAI model worked. Last error: {last_error}")

def ai_recommendations(seed, candidates, n=10):
    """Ask OpenAI to choose the best n artists from the Last.fm listener matches."""
    listing = "\n".join(f"- {c['name']} (listener overlap {round(c['match'] * 100)}%)"
                        for c in candidates[:40])
    result = ask_openai_json(
        "You are a music recommendation engine. You receive a band and a list of artists "
        "whose listeners overlap with that band's listeners, based on Last.fm listening data. "
        "Choose the best recommendations ONLY from that list. Favor strong overlap, but include "
        "a few less obvious picks so the list is not all side projects or near clones. "
        "For each pick, rate fit from 0 to 1 (how well the sound and style match) and write "
        "one specific sentence on why a fan of the band would enjoy it. "
        "Respond as JSON: {\"recommendations\": [{\"artist\": str, \"fit\": number, \"reason\": str}]}",
        f"Band: {seed['name']}\nGenre tags: {', '.join(seed['tags']) or 'unknown'}\n\n"
        f"Artists that the band's listeners also play:\n{listing}\n\n"
        f"Return {n} recommendations, best first.",
    )
    return result.get("recommendations", [])

In [6]:
# Step 6: The full recommendation pipeline
def recommend(band_name, n=10, verbose=True):
    log = print if verbose else (lambda *a, **k: None)

    seed = find_artist(band_name)
    if not seed:
        log(f"Last.fm could not find '{band_name}'. Check the spelling and try again.")
        return None
    log(f"Found on Last.fm: {seed['name']}  ({', '.join(seed['tags'][:4]) or 'no tags'})")

    candidates = get_similar_artists(seed["name"])
    if not candidates:
        log(f"Last.fm has no listener data for {seed['name']} yet. Try a more popular band.")
        return None
    log(f"Artists that {seed['name']} listeners also play: {len(candidates)}")

    picks = ai_recommendations(seed, candidates, n=n)
    by_name = {normalize(c["name"]): c for c in candidates}

    rows, used = [], {normalize(seed["name"])}
    for pick in picks:
        key = normalize(str(pick.get("artist", "")))
        cand = by_name.get(key)
        if not cand or key in used:   # skip anything not backed by listener data
            continue
        used.add(key)
        try:
            info = get_artist_info(cand["name"])
        except LastFMError:
            info = {"name": cand["name"], "url": cand["url"], "listeners": 0, "tags": []}
        try:
            fit = max(0.0, min(1.0, float(pick.get("fit", 0.5))))
        except (TypeError, ValueError):
            fit = 0.5
        rows.append({
            "Artist": info["name"],
            "Score": round(100 * (0.6 * cand["match"] + 0.4 * fit)),
            "Listener overlap %": round(cand["match"] * 100),
            "AI fit %": round(fit * 100),
            "Why": str(pick.get("reason", "")),
            "Tags": ", ".join(info["tags"][:3]),
            "Listeners": info["listeners"],
            "Last.fm": info["url"],
        })
        if len(rows) == n:
            break

    if not rows:   # fallback: OpenAI unavailable or returned nothing usable
        log("OpenAI picks could not be matched, showing the raw Last.fm listener matches.")
        rows = [{"Artist": c["name"], "Score": round(c["match"] * 100),
                 "Listener overlap %": round(c["match"] * 100), "AI fit %": None,
                 "Why": "", "Tags": "", "Listeners": None, "Last.fm": c["url"]}
                for c in candidates[:n]]

    df = pd.DataFrame(rows).sort_values("Score", ascending=False).reset_index(drop=True)
    df.index += 1
    return {"seed": seed, "results": df}

def show_results(output):
    if not output or output["results"].empty:
        display(HTML("<p>No recommendations found. Try another band.</p>"))
        return
    seed, df = output["seed"], output["results"]
    cards = []
    for rank, r in df.iterrows():
        spotify_link = "https://open.spotify.com/search/" + quote(r.Artist)
        listeners = f"{int(r.Listeners):,} listeners" if pd.notna(r.Listeners) and r.Listeners else ""
        details = " | ".join(x for x in [html.escape(r.Tags), listeners] if x)
        cards.append(f"""
        <div style="padding:10px 14px;margin:6px 0;border:1px solid #ddd;border-radius:10px;
                    font-family:sans-serif">
          <div style="font-size:16px;font-weight:600">{rank}.
            <a href="{html.escape(r['Last.fm'])}" target="_blank" style="color:#d51007;text-decoration:none">
            {html.escape(r.Artist)}</a>
            <span style="font-weight:400;color:#666;font-size:13px">
              score {r.Score} | {r['Listener overlap %']}% listener overlap</span>
            <a href="{spotify_link}" target="_blank"
               style="font-weight:400;font-size:12px;color:#1DB954;margin-left:8px">play on Spotify</a>
          </div>
          <div style="color:#888;font-size:12px">{details}</div>
          <div style="font-size:14px;margin-top:4px">{html.escape(r.Why)}</div>
        </div>""")
    display(HTML(f"<h3 style='font-family:sans-serif'>If you like {html.escape(seed['name'])}, "
                 f"try these:</h3>" + "".join(cards)))

## Try it

Type a band name and click **Recommend** (or press Enter). Results take about 10 to 20 seconds.

In [7]:
# Step 7: Interactive search box
import ipywidgets as widgets

band_box = widgets.Text(placeholder="e.g. Radiohead", description="Band:")
count_slider = widgets.IntSlider(value=10, min=3, max=15, description="How many:")
go_button = widgets.Button(description="Recommend", button_style="success")
output_area = widgets.Output()

def on_click(_=None):
    with output_area:
        output_area.clear_output()
        name = band_box.value.strip()
        if not name:
            print("Type a band name first.")
            return
        try:
            show_results(recommend(name, n=count_slider.value))
        except Exception as e:
            print("Something went wrong:", e)

go_button.on_click(on_click)
try:
    band_box.on_submit(on_click)   # Enter key (older ipywidgets)
except Exception:
    band_box.continuous_update = False
    band_box.observe(lambda change: on_click(), names="value")
display(widgets.VBox([widgets.HBox([band_box, go_button]), count_slider, output_area]))

### Optional: use it without the widget

If the search box does not display, run the cell below instead. It also shows the raw results table.